# ⚡ 用 vLLM-Hook 抽取表征，重跑 PII 层级探针

数据集、探针、融合、对照、词袋基线**全部沿用现有实验**，只把「从模型里取表征」这一步
从 HuggingFace 的 PyTorch forward hook 换成 [IBM/vLLM-Hook](https://github.com/IBM/vLLM-Hook)。

## 先说结论：能做，但要知道它解决的是什么问题

**能做的部分已核实**（读源码而非猜测）：

| 需求 | vLLM-Hook 是否满足 |
|---|---|
| 取**全部 36 层**，不只是最后一层 | ✅ 配置里 `"layers": [1..36]` 任意指定 |
| 每个词元的表征（好做均值池化） | ✅ `mode: "all_tokens"` 返回 `(seq_len, hidden)` |
| 末位词元表征（实验四那种） | ✅ `mode: "last_token"` 直接返回 `(hidden,)` |
| 只做前向、不真的生成 | ✅ 捕获发生在 **prefill** 阶段 |
| 离线批处理，不必起服务 | ✅ `HookLLM` 直接用 |
| 与 HuggingFace 的数值一致 | ⚠️ **理论上一致，必须实测**（见下） |

**一个关键的正确性细节**：vLLM 把残差加法融合进了下一层的归一化，所以它的 decoder block
返回的是 `(hidden_states, residual)`——**残差还没相加**。朴素捕获会得到与
HuggingFace `output_hidden_states` **不同**的张量。vLLM-Hook 在
`probe_hidden_states_worker.py` 里做了 `hidden = output[0] + output[1]` 把残差加回去，
层号也按 1-based 对齐。所以理论上一致——但这正是本 notebook 第一个 cell 要实测的东西。

## 但它不是当前的瓶颈

看实验一A 的实际耗时分布：

| 阶段 | 耗时 | 占比 |
|---|---|---|
| **抽取表征** | ~3 分钟 | **约 10%** |
| 逐层 L1 探针 | 2 分 33 秒 | 9% |
| **打乱标签对照探针** | **22 分 26 秒** | **80%** |

**抽取只占一成。** 换成 vLLM 最多省下两三分钟，而真正的瓶颈是对照探针的 L1 拟合，
那是 CPU 上的事，与推理引擎无关。

## 那什么时候它才真正有价值

1. **更大的模型 / 更多的数据**——吞吐优势随规模放大；
2. **生成时逐词元探测**——vLLM-Hook 在 decode 阶段同样挂钩，
   而 HF 那条路要做同样的事必须对每个前缀重跑一次模型；
3. **和真实的 vLLM 服务同栈**——如果守卫要部署在 vLLM 上，
   用同一套 hook 抽特征就不存在训练/推理两套代码的偏差。

第 2 点是这个项目此前搁置的方向，也是这条路径最有意思的地方。


## 步骤 1：环境

vLLM-Hook 的依赖是 `vllm>=0.5,<=0.21`、`torch>=2.0`，README 标注 Python 3.12。
先确认运行时是否满足——不满足的话后面全是白跑。

In [ ]:
!nvidia-smi
import sys, platform
print("Python:", platform.python_version())
try:
    import vllm; print("vLLM:", vllm.__version__)
except ImportError:
    print("vLLM: 未安装")


## 步骤 2：装依赖并同步仓库

vLLM 体积很大（数 GB），首次安装需要几分钟。

In [ ]:
import os
os.chdir('/content')

# 本项目
REPO = 'siren-pii-probing'
if os.path.isdir(REPO):
    !cd $REPO && git fetch -q origin && git reset -q --hard origin/main
else:
    !git clone -q https://github.com/jackyluo-learning/siren-pii-probing.git
!cd /content/$REPO && git log --oneline -1

# vLLM-Hook
if not os.path.isdir('/content/vLLM-Hook'):
    !git clone -q https://github.com/IBM/vLLM-Hook.git /content/vLLM-Hook
!pip install -q -r /content/vLLM-Hook/requirement.txt
!pip install -q -e /content/vLLM-Hook/vllm_hook_plugins
!pip install -q datasets scikit-learn matplotlib scipy

%cd /content/siren-pii-probing


## 步骤 3（必做）：数值一致性检验

**这一步不能跳过。** 两个推理引擎可以在 dtype、读取位置、残差是否相加、
padding 怎么屏蔽上各不相同——**每一项都会改变数值，而且都不会报错**。

这个 cell 取 16 条真实的任务文本，用两条路径各抽一次，逐层比对余弦相似度和相对差。

判读标准：

- fp16 与 fp32 的差异通常在 **1e-3** 量级，余弦相似度应 **> 0.999**；
- 若余弦明显低于 0.999，**不要继续**——先查 dtype、`max_model_len` 是否截断、层号是否对齐。

跑完会写出 `pii_vllm_parity.json`。

In [ ]:
!PYTHONPATH=.:examples python -u examples/run_pii_vllm.py \
    --task parity --model "Qwen/Qwen3-4B" --parity-n 16 --max-model-len 512


## 步骤 4：用 vLLM 路径跑实验一A

与 HuggingFace 版的实验一A **完全同一个任务**：同样的数据、同样的留出语料
（训练 dolly / 验证 ag_news / 测试 banking77）、同样的 C 网格与 η、同样的对照探针。
**唯一的差别是表征从哪来。**

所以结果可以直接和报告里的实验一A 并排读：

| | 实验一A（HuggingFace） |
|---|---|
| 词袋地板线 | 0.7851 |
| 最佳单层 | 0.9161（L4） |
| 跨层融合 | 0.8391 |
| 抽取耗时 | ~3 分钟 |

如果一致性检验通过，这一版的数字应该**非常接近**上表；差异大就说明抽取路径有问题，
而不是发现了什么新东西。

In [ ]:
!PYTHONPATH=.:examples python -u examples/run_pii_vllm.py \
    --task presence-merged --holdout-source banking77 \
    --model "Qwen/Qwen3-4B" --cap 6000 --per-source 1500 \
    --max-model-len 512 --chunk-size 32 --out-prefix pii_vllm

from IPython.display import Image, display
display(Image('pii_vllm_presence-merged_layers.png'))


## 步骤 5：末位词元池化（对应实验四）

`--pooling last` 走 vLLM-Hook 的 `last_token` 模式。

这里有个附带的好处：HF 那条路取末位词元必须按 `attention_mask` 定位真正的最后一个真实词元
（分词器右侧补齐，直接取 `[-1]` 会取到填充符，且不会报错）。
**vLLM 不做 padding**，`hidden[end-1]` 天然就是最后一个真实词元，这个陷阱不存在。

HuggingFace 版实验四的结果：最佳单层 0.8685（L5），跨层融合 0.6223。

In [ ]:
!PYTHONPATH=.:examples python -u examples/run_pii_vllm.py \
    --task presence-merged --holdout-source banking77 --pooling last \
    --model "Qwen/Qwen3-4B" --cap 6000 --per-source 1500 \
    --max-model-len 512 --chunk-size 32 --out-prefix pii_vllm

from IPython.display import Image, display
display(Image('pii_vllm_presence-merged_last_layers.png'))


## 步骤 6：两条路径并排对比

把 vLLM 版和 HuggingFace 版的逐层曲线画在一起。**它们应该几乎重合**——
这个 cell 的作用不是发现差异，而是确认没有差异。

In [ ]:
import json, glob, numpy as np, matplotlib.pyplot as plt

def load(pattern):
    hits = sorted(glob.glob(pattern))
    return json.load(open(hits[-1])) if hits else None

vl = load('pii_vllm_presence-merged_results.json')
hf = load('pii_presence_merged_ho_banking77_results.json')

if not (vl and hf):
    print("缺少结果文件。需要先跑本 notebook 的步骤 4，"
          "以及主 notebook 里 HuggingFace 版的实验一A。")
else:
    layers = sorted(int(k) for k in vl['test_f1'])
    a = np.array([hf['test_f1'][str(l)] for l in layers])
    b = np.array([vl['test_f1'][str(l)] for l in layers])
    fig, ax = plt.subplots(figsize=(8, 4.6), dpi=160)
    ax.plot(layers, a, label='HuggingFace forward hook', linewidth=2.2, color='#0E6A6E')
    ax.plot(layers, b, label='vLLM-Hook', linewidth=2.2, color='#B07A2E', linestyle='--')
    ax.axhline(hf['lexical_baseline'], color='#9A5518', linestyle=':', linewidth=1.4)
    ax.set_xlabel('Layer index'); ax.set_ylabel('Macro-F1')
    ax.set_title('Same task, same probes — only the extraction path differs')
    ax.grid(alpha=.3); ax.legend()
    plt.tight_layout(); plt.show()

    d = np.abs(a - b)
    print(f"逐层测试 Macro-F1 的绝对差：均值 {d.mean():.4f}  最大 {d.max():.4f}（L{layers[int(d.argmax())]}）")
    print(f"最佳单层   HF {a.max():.4f} (L{layers[int(a.argmax())]})  |  vLLM {b.max():.4f} (L{layers[int(b.argmax())]})")
    print(f"跨层融合   HF {hf['siren_test_f1']:.4f}  |  vLLM {vl['siren_test_f1']:.4f}")
    print(f"抽取耗时   vLLM {vl.get('extract_seconds', float('nan')):.0f}s")
    print()
    print("差异应在 0.01 量级以内（fp16 舍入 + 探针拟合的随机性）。"
          "若明显更大，回到步骤 3 的一致性检验找原因。")


## 已知限制

- **抽取不是瓶颈。** 本 notebook 优化的是整条流水线里约一成的耗时，
  真正的大头是打乱标签对照探针的 L1 拟合（CPU，与推理引擎无关）。
- **内存。** `all_tokens` 模式下每条文本要暂存 `36 层 × 序列长 × 2560 维`；
  fp16 下约 24 MB/条，所以 `--chunk-size` 默认只有 32。整个划分一次性跑会撑爆显存/共享内存。
- **Python 与 vLLM 版本。** README 标 Python 3.12、`vllm<=0.21`；
  Colab 运行时若不匹配，步骤 1 会先暴露出来。
- **本地未验证。** 编写这套代码的机器没有 CUDA，无法跑通 vLLM，
  所以步骤 3 的一致性检验是**必须**的，不是可选的。

## 下一步值得做的

真正能发挥 vLLM-Hook 长处的是**生成时逐词元探测**：让模型真的生成回复，
在 decode 的每一步取表征、打分，看标识符是在第几个词元被认出来的。
HF 那条路要做同样的事，必须对每个前缀重跑一次模型——O(T²)；
而 vLLM-Hook 在 decode 时本来就在挂钩，是 O(T)。
